In [ ]:
# https://www.freecodecamp.org/news/using-data-science-to-understand-what-makes-wine-taste-good-669b496c67ee
# https://machinelearningmastery.com/machine-learning-in-python-step-by-step/

GitHub repo with report: https://github.com/juliana-ng/Language-Learning-to-Fluency-Kaggle-Dataset

# Load Datasets & Libraries

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

pd.set_option("display.max_colwidth", None)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, linregress, ttest_ind, chi2_contingency
from matplotlib.gridspec import GridSpec

In [ ]:
path = kagglehub.dataset_download("sergionefedov/language-learning-to-fluency-who-gets-there")
df = pd.read_csv(os.path.join(path, "language_learning_fluency.csv"))
                 ,#index_col='learner_id')
df_dictionary = pd.read_csv(os.path.join(path, "data_dictionary_language.csv"))

In [ ]:
df_dictionary

# EDA

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df['learner_id'].nunique()

In [ ]:
df.head()

## Summary Statistics

In [ ]:
np.round(df.describe()).T

## Data distribution of variables

In [ ]:
print(df.groupby('reached_fluency').size())
print('\n')
print(df.groupby('cefr_level').size())
print('\n')
print(df.groupby('fsi_category').size())
print('\n')
print(df.groupby('prior_languages').size())

In [ ]:
total = df.shape[0]
not_fluent = np.round(len(df[df['reached_fluency'] == 0])*100/total)
fluent = np.round(len(df[df['reached_fluency'] == 1])*100/total)

print(f"""
{fluent}% of the learners are fluent
{not_fluent}% of the learners are not fluent""")


In [ ]:
x_var = [
    "reached_fluency",
     "fsi_category",
     "cefr_level",
     "prior_languages"
]

fig = plt.figure(figsize=(12, 6))
outer = GridSpec(2, 2, figure=fig, wspace=0.4, hspace=0.6)
for i, x_col in enumerate(x_var):
    temp = df[[x_col]].dropna()
    ax = fig.add_subplot(outer[i // 2, i % 2])

    order = None
    if x_col == 'cefr_level':
        order = ['A0', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2']

    sns.countplot(x=x_col,
          data=temp,
          color = 'hotpink',
          order = order,
          ax = ax
        )
    
    if x_col == 'reached_fluency':
        ax.set_xticks([0,1])
        ax.set_xticklabels(["Not Fluent", "Fluent"])
        xlab = x_col.replace("_", " ").title() 
        ax.set_xlabel(xlab, fontsize=9)
        ax.set_title(f'Distribution of {xlab}')
    elif x_col == 'fsi_category':
        ax.set_xticks([0,1,2,3])
        ax.set_xticklabels(["1", "2", "3", "4"])
        xlab = "FSI Category"
        ax.set_xlabel(xlab, fontsize=9)
        ax.set_title(f'Distribution of {xlab}')
    elif x_col == 'cefr_level':
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order)
        xlab = "CEFR Level"
        ax.set_xlabel(xlab, fontsize=9)
        ax.set_title(f'Distribution of {xlab}')
    elif x_col == 'prior_languages':
        ax.set_xticks(range(df['prior_languages'].nunique()))
        xlab = x_col.replace("_", " ").title() 
        ax.set_xlabel(xlab, fontsize=9)
        ax.set_title(f'Distribution of {xlab}')
            
    ax.set_ylabel('Count', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
#https://matplotlib.org/stable/gallery/color/named_colors.html
# https://seaborn.pydata.org/tutorial/color_palettes.html

# plt.rcParams['axes.titlesize'] = 10
# params = {'axes.titlesize':'10',
#           'xtick.labelsize':'8',
#           'ytick.labelsize':'8'}
# plt.rcParams.update(params)

# df.hist(figsize = (8,8), 
#         color = 'hotpink')
# plt.tight_layout()
# plt.show()

###############################################################################################

sns.set_style("white")

numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns

plt.figure(figsize=(14, len(numerical_columns) * 3))
for idx, feature in enumerate(numerical_columns, 1):
    plt.subplot(len(numerical_columns), 4, idx)
    sns.histplot(df[feature], kde=True, color = 'hotpink')
    plt.title(f"{feature} | Skewness: {round(df[feature].skew(), 2)}")

plt.tight_layout()
plt.show()

## Scatter Matrix

In [ ]:
pd.plotting.scatter_matrix(df, 
                           alpha = 0.3, 
                           figsize = (30,30), 
                           diagonal = 'kde',
                           color='hotpink',
                           hist_kwds={'color': 'hotpink'},
                           density_kwds={'color': 'hotpink'});

## One Hot Encoding

In [ ]:
col_to_cat = ['fsi_category', 'prior_languages']
for i in col_to_cat:
    df[i] = df[i].astype(str)
df.info()

In [ ]:
one_hot_encoded_df = pd.get_dummies(df.iloc[:,1:])
one_hot_encoded_df = one_hot_encoded_df.set_index(df["learner_id"])
one_hot_encoded_df.info()

In [ ]:
one_hot_encoded_df.head()

## Correlation Matrix

In [ ]:
#https://www.kaggle.com/code/dansbecker/using-categorical-data-with-one-hot-encoding
correlation = one_hot_encoded_df.iloc[:,1:].corr()
# display(correlation)
plt.figure(figsize=(24, 14))
heatmap = sns.heatmap(correlation, annot=True, linewidths=0, vmin=-1, cmap="RdBu_r")

## EDA

In [ ]:
# Define the three plot configurations
plots = [
    ("total_study_hours", "comprehensible_input_hours",
     "Total Study Hours", "Comprehensible Input Hours"),
    ("motivation", "consistency",
     "Motivation", "Consistency"),
    ("age_started", "accent_nativelike",
     "Age Started", "Native-like Accent"),
    ("active_use_share", "speaking_practice",
     "Active Use Share", "Speaking Practice")
]

fig = plt.figure(figsize=(22, 5))
outer = GridSpec(1, 4, figure=fig, wspace=0.4)

for i, (x_col, y_col, xlabel, ylabel) in enumerate(plots):

    temp = df[[x_col, y_col]].dropna()

    inner = outer[i].subgridspec(
        2, 2,
        width_ratios=[4, 1],
        height_ratios=[1, 4],
        wspace=0,
        hspace=0
    )

    ax_histx = fig.add_subplot(inner[0, 0])
    ax_joint = fig.add_subplot(inner[1, 0])
    ax_histy = fig.add_subplot(inner[1, 1])

    # Regression plot
    sns.regplot(
        data=temp, x=x_col, y=y_col,
        ax=ax_joint,
        scatter_kws={"s": 10, "color": "hotpink"},
        line_kws={"color": "lightpink"}        
    )

    # Marginal histograms
    sns.histplot(data=temp, x=x_col, ax=ax_histx, color="hotpink")
    sns.histplot(data=temp, y=y_col, ax=ax_histy, color="hotpink")

    for ax in [ax_histx, ax_histy]:
        for spine in ax.spines.values():
            spine.set_visible(False)
            
    # Regression statistics
    reg = linregress(temp[x_col], temp[y_col])

    stats_text = (
        f"Pearson r = {reg.rvalue:.3f}\n"
        f"R² = {reg.rvalue**2:.3f}\n"
        f"p = {reg.pvalue:.4g}"
    )

    ax_joint.text(
        0.05, 0.95, stats_text,
        transform=ax_joint.transAxes,
        verticalalignment="top",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.8)
    )

    ax_joint.set_xlabel(xlabel, fontsize=9)
    ax_joint.set_ylabel(ylabel, fontsize=9)

    ax_histx.set_title(f"{xlabel} vs. {ylabel}", fontsize=10)

    # Clean up marginal axes
    ax_histx.set_xlabel("")
    ax_histy.set_ylabel("")
    ax_histx.tick_params(labelbottom=False)
    ax_histy.tick_params(labelleft=False)

plt.show()

In [ ]:
temp = df[['total_study_hours', 'comprehensible_input_hours']]

x = temp['total_study_hours']
y = temp['comprehensible_input_hours']

from statsmodels.formula.api import ols
model = ols("comprehensible_input_hours ~ total_study_hours", temp).fit()
print(model.summary())


## Fluency, CEFR Level vs Hours, Active Use, FSI Category

In [ ]:
y_var = [
    "total_study_hours",
     "active_use_share",
     "fsi_category"
]

x_var = [
    "reached_fluency",
     "cefr_level"
]

fig = plt.figure(figsize=(12, 10))
outer = GridSpec(len(y_var), len(x_var), figure=fig, wspace=0.6, hspace=0.6)
# fig = plt.figure(figsize=(12, 8))
# outer = GridSpec(len(y_var), len(x_var), figure=fig, wspace=0.2, hspace=0.4)

for i, y_col in enumerate(y_var):
    for j, x_col in enumerate(x_var):
        temp = df[[y_col, x_col]].dropna()

        ax = fig.add_subplot(outer[i,j])

        ###############################################################################
        ###############################################################################
        if x_col == "reached_fluency":
            if y_col == "fsi_category":
                pct_df = pd.crosstab(
                    temp[y_col],
                    temp[x_col],
                    normalize="index"
                ).mul(100).round(2)
                
                pct_df = pct_df.reindex(columns=[0, 1], fill_value=0)
                
                pct_df.plot(
                    kind = "bar",
                    stacked = False,
                    ax = ax,
                    color = ['lightpink', "hotpink"]
                )
                ax.set_xticks(range(len(pct_df.index)))
                ax.set_xticklabels(pct_df.index)
                
                ax.set_ylim(0, 100)

                ax.tick_params(axis="x", labelrotation=0)
                ax.legend(["Not Fluent", "Fluent"],
                          title='Fluency',
                          loc = 'upper left',
                          bbox_to_anchor = (1, 0.95),
                          fontsize=8,
                          frameon = False,
                          facecolor="white",
                          framealpha = 0.8)
        ###############################################################################
            else:
                
                sns.violinplot(
                    x=x_col,
                    y=y_col,
                    data=temp,
                    hue= x_col,
                    palette = {
                    0: 'lightpink', 
                    1: 'hotpink'}, 
                    alpha=0.7,
                    legend = False,
                    ax = ax)
                    
                fluent = temp.loc[temp[x_col] == 1, y_col]
                not_fluent = temp.loc[temp[x_col] == 0, y_col]
                
                t_stat, p_value = ttest_ind(   
                fluent,
                not_fluent,
                equal_var = True,
                alternative = "two-sided"
                )
                
                stats_text = (
                f"fluent mean = {fluent.mean():.3f}\n"
                f"not fluent mean = {not_fluent.mean():.3f}\n"
                f"t statistic = {t_stat:.3f}\n"
                f"p-value = {p_value:.4g}"
                )
                
                ax.text(
                0.05, 0.95, stats_text,
                transform=ax.transAxes,
                verticalalignment="top",
                fontsize=8,
                bbox=dict(facecolor="white", alpha=0.8)
                )
    
                ax.set_xticks([0, 1])
                ax.set_xticklabels(["Not Fluent", "Fluent"])
        ###############################################################################
        ###############################################################################
        elif x_col == "cefr_level":            
            order = ['A0', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2']
            color = 'Pastel1'
            
            if y_col == "fsi_category":
                pct_df = pd.crosstab(
                        temp[x_col],
                        temp[y_col],
                        normalize="index"
                    ).mul(100).round(2)
                
                pct_df = pct_df.reindex(index=order)
                pct_df = pct_df.reindex(columns=[1, 2, 3, 4], fill_value=0)
                
                pct_df.plot(
                    kind = "bar",
                    stacked = False,
                    ax = ax,
                    color = sns.color_palette("Pastel1", 4)
                )
                # ax.set_xticks(range(len(pct_df.index)))
                # ax.set_xticklabels(pct_df.index)
                ax.set_ylim(0, 100)
                ax.tick_params(axis="x", labelrotation=0)
                ax.legend(['1', '2', '3', '4'],
                          title='FSI Category',                        
                          loc = 'upper left',
                          bbox_to_anchor = (1, 0.95),
                          fontsize=8,
                          frameon = False,
                          facecolor="white",
                          framealpha = 0.8)
        ###############################################################################
            else:
                sns.violinplot(
                    x=x_col,
                    y=y_col,
                    data=temp,
                    hue= x_col,
                    order = order,
                    palette = color,
                    alpha=0.7,
                    legend = False)
        ###############################################################################
        ###############################################################################
        if x_col == "cefr_level":
            xlab = "CEFR Level"
        else:
            xlab = x_col.replace("_", " ").title()
                       
        if y_col == "fsi_category":
            if x_col == "reached_fluency":
                xlab = "FSI Category"
                ylab = "Percentage"
                ax.set_title(f"Fluency by FSI Category", fontsize=10)
            elif x_col == "cefr_level":
                ylab = "Percentage"
                ax.set_title(f"CEFR Level by FSI Category", fontsize=10)
        else:
            ylab = y_col.replace("_", " ").title()
            ax.set_title(f"{xlab} vs. {ylab}", fontsize=10)

        ax.set_xlabel(xlab, fontsize=9)
        ax.set_ylabel(ylab, fontsize=9)  
        


plt.show()

In [ ]:
contingency_table = pd.crosstab(
    df['reached_fluency'],
    df['fsi_category']
)

print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square: {chi2:.3f}")
print("P-value:", p_value)

In [ ]:
cefr_mapping = {
    "A0": 0,
    "A1": 1,
    "A2": 2,
    "B1": 3,
    "B2": 4,
    "C1": 5,
    "C2": 6
}

df['cefr_numeric'] = df['cefr_level'].map(cefr_mapping)
print(df.groupby('fsi_category')['cefr_numeric'].skew())


In [ ]:
temp = df[df['cefr_level'].isin(['A0', 'A1', 'A2', 'B1'])]

temp = pd.crosstab(
                    temp['cefr_level'],
                    temp['dropped_out'],
                    normalize="index"
                ).mul(100).round(2)
temp

## CEFR Level, FSI Category vs Drop Out

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12, 4))
ax1 = axes[0]
ax2 = axes[1]

temp = df[df['reached_fluency'] == 0]
temp = pd.crosstab(
                    temp['cefr_level'],
                    temp['dropped_out'],
                    normalize="index"
                ).mul(100).round(2)
order = ['A0', 'A1', 'A2', 'B1']

temp = temp.reindex(order)
temp = temp.reindex(columns = [0, 1], fill_value = 0)
temp.plot(
    kind = "bar",
    stacked = False,
    color = ["lightpink", "hotpink"],
    ax = ax1
)

ax1.set_ylabel('Percentage')
ax1.set_ylim(0, 100)
ax1.set_xticks(range(len(order)))
ax1.set_xticklabels(order)
ax1.set_xlabel("CEFR Level", fontsize=9)

ax1.legend(["Stayed", "Quitted"],
title='Drop Out Status',
loc = 'upper left',
bbox_to_anchor = (1, 0.95),
fontsize=8,
frameon = False,
facecolor="white",
framealpha = 0.8)

ax1.set_title(f"Drop Out Status by CEFR Level of non-fluent learners", fontsize=10)
ax1.tick_params(axis = "x", labelrotation = 0)

counts = pd.crosstab(
    df['cefr_level'],
    df['dropped_out']
)
chi2, p_value, dof, expected = chi2_contingency(counts)
ax1. text(
    0.5, -0.35,
    f"Chi-square: {chi2:.3f}\nP-value:,{p_value}",
    transform = ax1.transAxes,
    ha = "center",
    fontsize = 9
)

#################################################################################

temp = df[df['reached_fluency'] == 0]
temp = pd.crosstab(
                    temp['fsi_category'],
                    temp['dropped_out'],
                    normalize="index"
                ).mul(100).round(2)

temp = temp.reindex(columns = [0, 1], fill_value = 0)
temp.plot(
    kind = "bar",
    stacked = False,
    color = ["lightpink", "hotpink"],
    ax = ax2
)

ax2.set_ylabel('Percentage')
ax2.set_ylim(0, 100)
ax2.set_xlabel("FSI Category", fontsize=9)

ax2.legend(["Stayed", "Quitted"],
title='Drop Out Status',
loc = 'upper left',
bbox_to_anchor = (1, 0.95),
fontsize=8,
frameon = False,
facecolor="white",
framealpha = 0.8)

ax2.set_title(f"Drop Out Status by FSI Category of non-fluent learners", fontsize=10)
ax2.tick_params(axis = "x", labelrotation = 0)

counts = pd.crosstab(
    df['fsi_category'],
    df['dropped_out']
)
chi2, p_value, dof, expected = chi2_contingency(counts)
ax2. text(
    0.5, -0.35,
    f"Chi-square: {chi2:.3f}\nP-value: {p_value}",
    transform = ax2.transAxes,
    ha = "center",
    fontsize = 9
)
plt.tight_layout()
plt.show()

## Other variables vs Drop Out

In [ ]:
y_var = [
    "total_study_hours",
     "active_use_share",
     "motivation",
     "consistency"
]

x_var = [
    "dropped_out"
]

fig = plt.figure(figsize=(12, 6))
outer = GridSpec(2, 2, figure=fig, wspace=0.2, hspace=0.4)
for i, y_col in enumerate(y_var):
    for j, x_col in enumerate(x_var):
        temp = df[[y_col, x_col]].dropna()
        ax = fig.add_subplot(outer[i // 2, i % 2])
        sns.violinplot(x=x_col,
                       y=y_col,
                       data=temp,
                       hue= x_col,
                       palette = { 0: 'lightpink', 
                                   1: 'hotpink'},
                       alpha=0.7,
                       legend = False)
        
        fluent = temp.loc[temp[x_col] == 1, y_col]
        not_fluent = temp.loc[temp[x_col] == 0, y_col]
        
        t_stat, p_value = ttest_ind(   
        fluent,
        not_fluent,
        equal_var = True,
        alternative = "two-sided"
        )
        
        stats_text = (
        f"quitted mean = {fluent.mean():.3f}\n"
        f"stayed mean = {not_fluent.mean():.3f}\n"
        f"t statistic = {t_stat:.3f}\n"
        f"p-value = {p_value:.4g}"
        )
        
        ax.text(
        0.05, 0.95, stats_text,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.8)
        )

        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Stayed", "Quitted"])
        xlab = "Drop Out Status"
        ylab = y_col.replace("_", " ").title() 
        ax.set_ylabel(ylab, fontsize=9)
        ax.set_xlabel(xlab, fontsize=9)     
        ax.set_title(f"{xlab} vs. {ylab}", fontsize=10)

## Fluency and Drop Out

In [ ]:
y_var = [
    "feedback_quality",
     "speaking_practice",
     "comprehensible_input_hours"
]

x_var = [
    "reached_fluency",
     "dropped_out"
]

fig = plt.figure(figsize=(12, 10))
outer = GridSpec(len(y_var), len(x_var), figure=fig, wspace=0.6, hspace=0.6)

for i, y_col in enumerate(y_var):
    for j, x_col in enumerate(x_var):
        temp = df[[y_col, x_col]].dropna()

        ax = fig.add_subplot(outer[i,j])
                        
        sns.violinplot(x=x_col,
                       y=y_col,
                       data=temp,
                       hue= x_col,
                       palette = {
                       0: 'lightpink', 
                       1: 'hotpink'}, 
                       alpha=0.7,
                       legend = False,
                       ax = ax)
            


        fluent = temp.loc[temp[x_col] == 1, y_col]
        not_fluent = temp.loc[temp[x_col] == 0, y_col]
        
        t_stat, p_value = ttest_ind(   
        fluent,
        not_fluent,
        equal_var = True,
        alternative = "two-sided"
        )
        
        if x_col == "reached_fluency":
            fluent_text = "fluent"
            not_fluent_text = "not fluent"
            ax.set_xticks([0, 1])
            ax.set_xticklabels(["Not Fluent", "Fluent"])
            xlab = x_col.replace("_", " ").title()


        
        elif x_col == "dropped_out":
            fluent_text = "quitted"
            not_fluent_text = "stayed"
            ax.set_xticks([0, 1])
            ax.set_xticklabels(["Stayed", "Quitted"])
            xlab = "Drop Out Status"

        stats_text = (
        f"{fluent_text} mean = {fluent.mean():.3f}\n"
        f"{not_fluent_text} mean = {not_fluent.mean():.3f}\n"
        f"t statistic = {t_stat:.3f}\n"
        f"p-value = {p_value:.4g}"
        )
        
        ax.text(
        0.05, 0.95, stats_text,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.8)
        )

        ylab = y_col.replace("_", " ").title() 
        ax.set_xlabel(xlab, fontsize=9)
        ax.set_ylabel(ylab, fontsize=9)     
        ax.set_title(f"{xlab} vs. {ylab}", fontsize=10)

In [ ]:
cefr_mapping = {
    "A0": 0,
    "A1": 1,
    "A2": 2,
    "B1": 3,
    "B2": 4,
    "C1": 5,
    "C2": 6
}

corr, p_value = spearmanr(
    df["prior_languages"],
    df["cefr_level"].map(cefr_mapping),
    nan_policy="omit"
)

print(f"Spearman correlation: {corr:.3f}")
print(f"P-value: {p_value:.4g}")

In [ ]:
temp = df[["total_study_hours","dropped_out"]]
temp[temp["dropped_out"]==0]['total_study_hours'].median()


In [ ]:
temp[temp["dropped_out"]==1]['total_study_hours'].median()

In [ ]:
temp = df[["total_study_hours","reached_fluency"]]
temp[temp["reached_fluency"]==0]['total_study_hours'].median()

In [ ]:
temp[temp["reached_fluency"]==1]['total_study_hours'].median()